In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# Add project source directory to import path.
import sys
import os
from pathlib import Path

parent_dir = str(os.path.join(Path.cwd().parent, 'src'))

if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

# Imports.
from typing import cast
import math
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import make_interp_spline
import fvopn_simulations as fvopn

In [ ]:
rng = np.random.default_rng(42)
alpha = 2 # Root slots.
k = 2 # Multiplication factor.
size = 1000 # Number of nodes.
budgets=rng.beta(.5, 8, size) # Random generated budgets.
                    

conf = fvopn.VoteAuctionModelV2Config(
    budgets=budgets,
    root_slots=alpha,
    multiplication_factor=k,
)

model = fvopn.VoteAuctionModelV2(conf, rng)

In [ ]:
def graph_fn(up_to: int, fn):
    fn_vec = np.vectorize(fn)

    xs = np.arange(1, up_to)
    ys = fn_vec(xs)

    return (xs, ys)

In [ ]:
def V(whale_budget: float, x: int):
    b = np.copy(budgets)

    b[0:x] = whale_budget / x
    
    conf = fvopn.VoteAuctionModelV2Config(
        budgets=b,
        root_slots=alpha,
        multiplication_factor=k,
    )

    model = fvopn.VoteAuctionModelV2(conf, rng)

    res = 0

    N = 1
    
    for _ in range(N):
        a = model.run()

        a = a.sort_index()

        dBudget = 1+abs(whale_budget-a.loc[0:x, 'budget'].sum())
                      
        res += a.loc[0:x, 'votes'].sum() / dBudget

    res /= N

    return res

In [ ]:
V(10, 2)

In [ ]:
up_to = 20

x1, y1 = graph_fn(up_to, lambda x: V(np.max(budgets[x:])*5, x))
x2, y2 = graph_fn(up_to, lambda x: V(np.max(budgets[x:])*10, x))

In [ ]:
p1 = np.polynomial.polynomial.Polynomial.fit(x1, y1, 1).convert().coef
p1 = np.pad(p1, (0, 2 - len(p1)))
b1, m1 = p1

p2 = np.polynomial.polynomial.Polynomial.fit(x2, y2, 1).convert().coef
p2 = np.pad(p2, (0, 2 - len(p2)))
b2, m2 = p2

In [ ]:
mean1color = "tomato"
mean2color = "slateblue"

plt.plot(x1, y1, label="5*maximo", color=mean1color)
plt.plot(np.arange(1, up_to), np.arange(1, up_to) * m1 + b1, color=mean1color)
plt.plot(x2, y2, label="10*maximo", color=mean2color)
plt.plot(np.arange(1, up_to), np.arange(1, up_to) * m2 + b2, color=mean2color)
plt.xticks(np.arange(1, up_to, 5))
plt.title("Resistencia a Fraude")
plt.xlabel("Identidades")
plt.ylabel("Influencia")
plt.grid()
plt.legend()
plt.show()